# 🔬 Level 6: Principal Component Analysis (PCA)

**[📖 Want a detailed explanation? Read the Manual (Streamlit App)](https://bookseal-seoul-apt-price-prediction.streamlit.app/Level_6_PCA)**

In Level 5, we saw that too many features can be bad.
But we want to keep the useful info!

**PCA** compresses High-Dimensional data into Low-Dimensional data (2D or 3D) while keeping the important patterns.

### 💡 Mental Model: The Teapot Shadow

Imagine a 3D **Teapot**. You want to send a photo of it (2D) to a friend.
- If you take the photo from the **top**, it looks like a circle. (Bad - lost handle/spout)
- If you take it from the **side**, you see the handle and spout. (Good - preserved variance)

**PCA finds the best angle to take the photo** so you lose the least amount of information.

### 1. Load & Prepare High-Dimensional Data
Let's create the same 30+ features from Level 5.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

url = "https://github.com/bookseal/seoul-apt-price-prediction/raw/main/data/sample.parquet"
df = pd.read_parquet(url)

np.random.seed(42)
# Create synthetic extra features
if 'year' not in df.columns: df['year'] = np.random.randint(1985, 2024, len(df))
df['building_age'] = 2024 - df['year']
df['total_units'] = np.random.randint(100, 2000, len(df))
df['parking_ratio'] = np.random.uniform(0.5, 2.0, len(df))

# Prepare X
numeric = df[['area_m2', 'year', 'building_age', 'total_units', 'parking_ratio']].values
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
districts = encoder.fit_transform(df[['district']])

X = np.hstack([numeric, districts])
y = df['price_10k_krw'].values

print(f"Original Feature Count: {X.shape[1]}")

### 2. Standardization
PCA requires data to be centered and scaled.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 3. Apply PCA
Let's compress ~30 dimensions down to just **2 dimensions**.

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"Compressed Shape: {X_pca.shape}")
print(f"Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Variance Preserved: {sum(pca.explained_variance_ratio_):.2%}")

### 4. Visualize in 2D
Now we can plot all our data on a simple X-Y chart!

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', alpha=0.5, s=15)
plt.colorbar(label='Price')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('All Features Compressed to 2D')
plt.show()

### 5. Does it still work for prediction?
Let's train a model on the 2D data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
print(f"Model RMSE using only 2 PC features: {rmse:,.0f}")
print("Compare this to full model. We lost some info, but kept a lot!")